# 1. Loading the Datasets

In [ ]:
import pandas as pd
import os
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

In [ ]:
sample_sub = pd.read_csv("/kaggle/input/datasets/ikshugupta/nlp-projet/Sample.csv")
test = pd.read_csv("/kaggle/input/datasets/ikshugupta/nlp-projet/test (1).csv")
train = pd.read_csv("/kaggle/input/datasets/ikshugupta/nlp-projet/train (1).csv")

# 2. Exploratory Data Analysis (EDA)

## 2.1 - Basic Information

In [ ]:
print("Shape of train:", train.shape)
print("Shape of test:", test.shape)
print("Shape of sample_sub:", sample_sub.shape)

print("\nColumn names and data types:")
print(train.dtypes)

print("\nFirst 5 rows:")
train.head()

## 2.2 - Target Variable Analysis

In [ ]:
print("Label distribution (count):")
print(train['label'].value_counts())

print("\nLabel distribution (percentage):")
print(train['label'].value_counts(normalize=True)*100)

# Plot
plt.figure(figsize=(8,4))
ax = sns.countplot(x='label', data=train, hue='label', palette='viridis', legend=False)
plt.title('Label Distribution')
plt.xlabel('Label')
plt.ylabel('Count')

for p in ax.patches:
    ax.annotate(f'{int(p.get_height())}', 
                (p.get_x() + p.get_width()/2, p.get_height()),
                ha='center', va='bottom', fontsize=11)
plt.show()

Insight:
- Label 0 is the most common (57%) → model will be biased towards this
- Label 3 is the rarest (2.7%)     → model will struggle to predict this
- This is called CLASS IMBALANCE

## 2.3 - Missing Values Analysis

In [ ]:
print("Missing values in train:")
missing_train = train.isnull().sum()
missing_train = missing_train[missing_train > 0]
print(missing_train)
print("\nMissing percentage:")
print((missing_train/len(train)*100).round(2))

print("\nMissing values in test:")
missing_test = test.isnull().sum()
missing_test = missing_test[missing_test > 0]
print(missing_test)

# Plot
plt.figure(figsize=(8,4))
missing_train.plot(kind='bar', color='coral', edgecolor='black')
plt.title('Missing Values in Train')
plt.ylabel('Count')
plt.xticks(rotation=0)
plt.show()

## 2.4 - Numerical Features Analysis

In [ ]:
num_cols = ['upvote', 'downvote', 'if_1', 'if_2']

# Distribution of each numerical feature
fig, axes = plt.subplots(2, 2, figsize=(12,8))
for i, col in enumerate(num_cols):
    ax = axes[i//2][i%2]
    train[col].hist(bins=30, ax=ax, color='steelblue', edgecolor='black')
    ax.set_title(f'Distribution of {col}')
    ax.set_xlabel(col)
    ax.set_ylabel('Count')
plt.suptitle('Numerical Features Distribution')
plt.tight_layout()
plt.show()

# Relationship with label
fig, axes = plt.subplots(2, 2, figsize=(12,8))
for i, col in enumerate(num_cols):
    ax = axes[i//2][i%2]
    sns.boxplot(x='label', y=col, data=train, ax=ax, 
                hue='label', palette='Set2', legend=False)
    ax.set_title(f'{col} vs Label')
plt.suptitle('Numerical Features vs Label')
plt.tight_layout()
plt.show()

# Summary statistics
print("Summary statistics of numerical features:")
print(train[num_cols].describe())

## 2.5 - Categorical Features Analysis

In [ ]:
# Emoticons
emoticon_cols = ['emoticon_1', 'emoticon_2', 'emoticon_3']
fig, axes = plt.subplots(1, 3, figsize=(14,4))
for i, col in enumerate(emoticon_cols):
    train.groupby('label')[col].mean().plot(
        kind='bar', ax=axes[i], color='mediumpurple', edgecolor='black')
    axes[i].set_title(f'Average {col} per Label')
    axes[i].set_xlabel('Label')
    axes[i].set_ylabel('Mean Value')
    axes[i].tick_params(axis='x', rotation=0)
plt.suptitle('Emoticon Features vs Label')
plt.tight_layout()
plt.show()

# Race, Religion, Gender
cat_cols = ['race', 'religion', 'gender', 'disability']
fig, axes = plt.subplots(2, 2, figsize=(14,10))
for i, col in enumerate(cat_cols):
    ax = axes[i//2][i%2]
    train[col].fillna('missing').value_counts().plot(
        kind='bar', ax=ax, color='teal', edgecolor='black')
    ax.set_title(f'{col} value counts')
    ax.set_xlabel(col)
    ax.set_ylabel('Count')
    ax.tick_params(axis='x', rotation=45)
plt.suptitle('Categorical Features Distribution')
plt.tight_layout()
plt.show()

# How race/religion/gender relate to label
fig, axes = plt.subplots(1, 3, figsize=(16,5))
for i, col in enumerate(['race', 'religion', 'gender']):
    crosstab = pd.crosstab(train[col].fillna('missing'), train['label'], normalize='index')*100
    crosstab.plot(kind='bar', ax=axes[i], colormap='viridis', edgecolor='black')
    axes[i].set_title(f'{col} vs Label (%)')
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Percentage')
    axes[i].tick_params(axis='x', rotation=45)
    axes[i].legend(title='label')
plt.suptitle('Categorical Features vs Label')
plt.tight_layout()
plt.show()

## 2.6 - Text Feature Analysis

In [ ]:
# Add text features
train['comment'] = train['comment'].fillna('')
train['comment_length'] = train['comment'].apply(len)
train['word_count'] = train['comment'].apply(lambda x: len(x.split()))

# Comment length distribution
plt.figure(figsize=(10,4))
sns.histplot(data=train, x='comment_length', hue='label', 
             bins=50, palette='tab10')
plt.title('Comment Length Distribution by Label')
plt.xlabel('Comment Length')
plt.xlim(0, 2000)
plt.show()

# Word count distribution  
plt.figure(figsize=(10,4))
sns.histplot(data=train, x='word_count', hue='label',
             bins=50, palette='tab10')
plt.title('Word Count Distribution by Label')
plt.xlabel('Word Count')
plt.xlim(0, 300)
plt.show()

# Average comment length per label
plt.figure(figsize=(7,4))
train.groupby('label')['comment_length'].mean().plot(
    kind='bar', color='orange', edgecolor='black')
plt.title('Average Comment Length per Label')
plt.xlabel('Label')
plt.ylabel('Average Length')
plt.xticks(rotation=0)
plt.show()

print("Average comment length per label:")
print(train.groupby('label')['comment_length'].mean().round(2))
print("\nAverage word count per label:")
print(train.groupby('label')['word_count'].mean().round(2))

## 2.7 - Date Feature Analysis

In [ ]:
# Extract date features
train['created_date'] = pd.to_datetime(train['created_date'], utc=True)
train['hour']       = train['created_date'].dt.hour
train['month']      = train['created_date'].dt.month
train['dayofweek']  = train['created_date'].dt.dayofweek
train['year']       = train['created_date'].dt.year

# Hour vs label
plt.figure(figsize=(12,4))
sns.countplot(x='hour', hue='label', data=train, palette='tab10')
plt.title('Comments by Hour of Day')
plt.xlabel('Hour')
plt.ylabel('Count')
plt.show()

# Day of week vs label
plt.figure(figsize=(10,4))
sns.countplot(x='dayofweek', hue='label', data=train, palette='tab10')
plt.title('Comments by Day of Week (0=Monday, 6=Sunday)')
plt.xlabel('Day of Week')
plt.ylabel('Count')
plt.show()

# Month vs label
plt.figure(figsize=(12,4))
sns.countplot(x='month', hue='label', data=train, palette='tab10')
plt.title('Comments by Month')
plt.xlabel('Month')
plt.ylabel('Count')
plt.show()

## 2.8 - Correlation Analysis

In [ ]:
# Correlation heatmap
corr_cols = ['upvote', 'downvote', 'if_1', 'if_2', 
             'emoticon_1', 'emoticon_2', 'emoticon_3',
             'comment_length', 'word_count', 'hour', 
             'month', 'dayofweek', 'label']

plt.figure(figsize=(12,8))
sns.heatmap(train[corr_cols].corr(), 
            annot=True, fmt='.2f', 
            cmap='coolwarm',
            center=0)
plt.title('Correlation Heatmap')
plt.tight_layout()
plt.show()

# Top features correlated with label
print("Features most correlated with label:")
corr_with_label = train[corr_cols].corr()['label'].drop('label')
print(corr_with_label.abs().sort_values(ascending=False))

# 3. Data Preprocessing

## 3.1 - Fill Missing Values

In [ ]:
# Fill comment
train['comment'] = train['comment'].fillna('')
test['comment']  = test['comment'].fillna('')

# Fill race, religion, gender with 'missing' as a new category
for col in ['race', 'religion', 'gender']:
    train[col] = train[col].fillna('missing')
    test[col]  = test[col].fillna('missing')

print("Missing values after filling:")
print(train.isnull().sum())
print("\nTest missing values after filling:")
print(test.isnull().sum())

## 3.2 - Extract Date Features

In [ ]:
train['created_date'] = pd.to_datetime(train['created_date'], utc=True)
test['created_date']  = pd.to_datetime(test['created_date'],  utc=True)

for df in [train, test]:
    df['hour']      = df['created_date'].dt.hour
    df['month']     = df['created_date'].dt.month
    df['dayofweek'] = df['created_date'].dt.dayofweek
    df['year']      = df['created_date'].dt.year
    df['quarter']   = df['created_date'].dt.quarter

print("Date features added:")
print(train[['hour', 'month', 'dayofweek', 'year', 'quarter']].head())

## 3.3 - Extract Text Features

In [ ]:
for df in [train, test]:
    df['comment_length']  = df['comment'].apply(len)
    df['word_count']      = df['comment'].apply(lambda x: len(x.split()))
    df['unique_words']    = df['comment'].apply(lambda x: len(set(x.split())))
    df['avg_word_length'] = df['comment'].apply(
        lambda x: np.mean([len(w) for w in x.split()]) if len(x.split()) > 0 else 0)
    df['uppercase_count'] = df['comment'].apply(
        lambda x: sum(1 for c in x if c.isupper()))
    df['exclamation']     = df['comment'].apply(lambda x: x.count('!'))
    df['question']        = df['comment'].apply(lambda x: x.count('?'))
    df['punctuation']     = df['comment'].apply(
        lambda x: sum(1 for c in x if c in '!?.,;:'))
    df['uppercase_ratio'] = df['uppercase_count'] / (df['comment_length'] + 1)
    df['lexical_diversity'] = df['unique_words'] / (df['word_count'] + 1)

print("Text features added:")
print(train[['comment_length', 'word_count', 'unique_words', 
             'avg_word_length', 'uppercase_count', 'exclamation',
             'question', 'punctuation', 'uppercase_ratio',
             'lexical_diversity']].head())

## 3.4 - Handle Outliers

In [ ]:
# Cap extreme values at 99th percentile

for col in ['upvote', 'downvote', 'if_1', 'if_2']:
    cap = train[col].quantile(0.99)
    train[col] = train[col].clip(upper=cap)
    test[col]  = test[col].clip(upper=cap)
    print(f"{col} capped at: {cap}")

##  3.5 - Encode Categorical Features

In [ ]:
from sklearn.preprocessing import LabelEncoder

# Encode race, religion, gender
for col in ['race', 'religion', 'gender']:
    le = LabelEncoder()
    # fit on both train and test combined
    # so no unseen categories cause errors
    combined = pd.concat([train[col], test[col]])
    le.fit(combined)
    train[col] = le.transform(train[col])
    test[col]  = le.transform(test[col])
    print(f"{col} encoded. Unique values: {train[col].unique()}")

# Convert disability boolean to integer
train['disability'] = train['disability'].astype(int)
test['disability']  = test['disability'].astype(int)

print("\nEncoding done!")
print(train[['race', 'religion', 'gender', 'disability']].head())

## 3.6 - Select Final Features

In [ ]:
# Select all features for modeling
feature_cols = [
    # original numerical features
    'emoticon_1', 'emoticon_2', 'emoticon_3',
    'upvote', 'downvote', 
    'if_1', 'if_2',
    # categorical encoded
    'race', 'religion', 'gender', 'disability',
    # text features
    'comment_length', 'word_count', 'unique_words',
    'avg_word_length', 'uppercase_count', 'exclamation',
    'question', 'punctuation', 'uppercase_ratio',
    'lexical_diversity',
    # date features
    'hour', 'month', 'dayofweek', 'year', 'quarter'
]

X = train[feature_cols]
y = train['label']
X_test_final = test[feature_cols]

print("Features shape:", X.shape)
print("Target shape:", y.shape)
print("Test shape:", X_test_final.shape)
print("\nFeature list:")
print(feature_cols)

## 3.7 - Scale Features

In [ ]:
from sklearn.preprocessing import StandardScaler

# Scale numerical features
scaler = StandardScaler()
X_scaled       = scaler.fit_transform(X)
X_test_scaled  = scaler.transform(X_test_final)

# Convert back to dataframe for easier use
X_scaled      = pd.DataFrame(X_scaled,      columns=feature_cols)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=feature_cols)

print("Scaling done!")
print(X_scaled.describe().round(2))

In [ ]:
print(feature_cols)

In [ ]:
print(X[feature_cols].nunique().sort_values())

# 4. Feature Engineering

## 4.1 - Apply TF-IDF

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import hstack, csr_matrix

# Apply TF-IDF on comment text
print("Fitting TF-IDF...")
tfidf = TfidfVectorizer(
    max_features=9000,    # use top 9000 most important words
    ngram_range=(1, 1),   # single words
    min_df=2,             # ignore words appearing less than 2 times
    strip_accents='unicode',
    sublinear_tf=True     # reduces effect of very common words
)

X_tfidf_train = tfidf.fit_transform(train['comment'])
X_tfidf_test  = tfidf.transform(test['comment'])

print("TF-IDF train shape:", X_tfidf_train.shape)
print("TF-IDF test shape:",  X_tfidf_test.shape)

## 4.2 - Combine TF-IDF with Numerical Features

In [ ]:
X_num_train = csr_matrix(X_scaled.values)
X_num_test  = csr_matrix(X_test_scaled.values)

X_combined_train = hstack([X_num_train, X_tfidf_train])
X_combined_test  = hstack([X_num_test,  X_tfidf_test])

print("Combined train shape:", X_combined_train.shape)
print("Combined test shape:",  X_combined_test.shape)

# 5. Train Validation Spliting

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    X_combined_train, y,
    test_size=0.2,
    random_state=42,
    stratify=y        # ensures same label distribution in both splits
)

print("X_train shape:", X_train.shape)
print("X_val shape:",   X_val.shape)
print("y_train shape:", y_train.shape)
print("y_val shape:",   y_val.shape)

print("\nLabel distribution in y_train:")
print(y_train.value_counts(normalize=True).round(4)*100)
print("\nLabel distribution in y_val:")
print(y_val.value_counts(normalize=True).round(4)*100)

# 6. Models

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from xgboost import XGBClassifier
from sklearn.metrics import f1_score, classification_report

## 6.1 - Logistic Regression

In [ ]:
print("Training Logistic Regression...")
lr = LogisticRegression(
    solver='saga',
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)
lr.fit(X_train, y_train)
lr_preds = lr.predict(X_val)
lr_f1 = f1_score(y_val, lr_preds, average='macro')
print(f"LR Macro F1: {lr_f1:.4f}")
print(classification_report(y_val, lr_preds))

## 6.2 - Linear SVM

In [ ]:
print("\nTraining Linear SVM...")
svm = LinearSVC(
    class_weight='balanced',
    random_state=42
)
svm.fit(X_train, y_train)
svm_preds = svm.predict(X_val)
svm_f1 = f1_score(y_val, svm_preds, average='macro')
print(f"SVM Macro F1: {svm_f1:.4f}")
print(classification_report(y_val, svm_preds))

## 6.3 - XGBoost

In [ ]:
print("\nTraining XGBoost...")
xgb = XGBClassifier(
    eval_metric='mlogloss',
    random_state=42,
    n_jobs=-1
)
xgb.fit(X_train, y_train)
xgb_preds = xgb.predict(X_val)
xgb_f1 = f1_score(y_val, xgb_preds, average='macro')
print(f"XGB Macro F1: {xgb_f1:.4f}")
print(classification_report(y_val, xgb_preds))

In [ ]:
feature_names = list(X_scaled.columns) + list(tfidf.get_feature_names_out())

importance = pd.Series(
    xgb.feature_importances_,
    index=feature_names
)

top_30 = importance.sort_values(ascending=False).head(30)

print("Top 30 XGBoost Feature Importances:\n")
print(top_30)

plt.figure(figsize=(10, 10))
top_30.sort_values().plot(kind='barh')
plt.xlabel("Feature Importance")
plt.ylabel("Feature")
plt.title("Top 30 XGBoost Feature Importances")
plt.tight_layout()
plt.show()

numeric_importance = importance[X_scaled.columns]
text_importance = importance[tfidf.get_feature_names_out()]

print("Top numerical features:")
print(numeric_importance.sort_values(ascending=False).head(20))

print("\nTop text features:")
print(text_importance.sort_values(ascending=False).head(30))

## 6.4 - LightGBM

In [ ]:
from lightgbm import LGBMClassifier

lgbm = LGBMClassifier(
    objective='multiclass',
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=31,
    random_state=42,
    n_jobs=-1,
    verbosity=-1
)

lgbm.fit(X_train, y_train)

lgbm_preds = lgbm.predict(X_val)

lgbm_f1 = f1_score(
    y_val,
    lgbm_preds,
    average='macro'
)

print(f"LightGBM Macro F1: {lgbm_f1:.4f}")
print(classification_report(y_val, lgbm_preds))

In [ ]:
feature_names = list(X_scaled.columns) + list(tfidf.get_feature_names_out())

importance = pd.Series(
    lgbm.feature_importances_,
    index=feature_names
)

top_30 = importance.sort_values(ascending=False).head(30)

print(top_30)

plt.figure(figsize=(10, 10))
top_30.sort_values().plot(kind='barh')
plt.xlabel("Feature Importance")
plt.ylabel("Feature")
plt.title("Top 30 LightGBM Feature Importances")
plt.tight_layout()
plt.show()

numeric_importance = importance[X_scaled.columns]
text_importance = importance[tfidf.get_feature_names_out()]

print("Top numerical features:")
print(numeric_importance.sort_values(ascending=False).head(20))

print("\nTop text features:")
print(text_importance.sort_values(ascending=False).head(30))

## 6.5 - Ensemble (XGBoost + LightGBM)

In [ ]:
xgb_proba = xgb.predict_proba(X_val)
lgbm_proba = lgbm.predict_proba(X_val)

ensemble_proba = 0.3 * xgb_proba + 0.7 * lgbm_proba

ensemble_preds = np.argmax(ensemble_proba, axis=1)

ensemble_f1 = f1_score(
    y_val,
    ensemble_preds,
    average='macro'
)

print("Ensemble Macro F1:", ensemble_f1)

# 7. Model & Plot Comparission

In [ ]:
print("\n--- Model Comparison (Macro F1) ---")
results = pd.DataFrame({
    'Model': ['Logistic Regression', 'Linear SVM', 'XGBoost', 'LightGBM', 'Ensemble'],
    'Macro F1': [lr_f1, svm_f1, xgb_f1, lgbm_f1, ensemble_f1]
})
print(results.sort_values('Macro F1', ascending=False))

# 8. Submission

In [ ]:
best_model_name = results.loc[results['Macro F1'].idxmax(), 'Model']

if best_model_name == 'Ensemble':
    test_proba = 0.3 * xgb.predict_proba(X_combined_test) + 0.7 * lgbm.predict_proba(X_combined_test)
    test_predictions = np.argmax(test_proba, axis=1)
else:
    best_model = {
        'Logistic Regression': lr,
        'Linear SVM': svm,
        'XGBoost': xgb,
        'LightGBM': lgbm
    }[best_model_name]
    test_predictions = best_model.predict(X_combined_test)


print("Best Model:", best_model_name)

In [ ]:
sample_sub['label'] = test_predictions
sample_sub.to_csv('submission.csv', index=False)
print("Submission saved!")
print(sample_sub.head())
print("\nPrediction distribution:")
print(sample_sub['label'].value_counts())